# Notebook 01 — Extract (MIMIC-IV Demo) & Raw preview

Objectif : vérifier que l’extraction fonctionne et comprendre rapidement les données brutes (`raw`).

Fichiers importants :
- `data/raw/mimic_iv_demo/...` : tables MIMIC (hosp/icu)
- `data/raw/patient_vitals_mimic_demo_raw.csv` : ton fichier raw “projet” construit depuis MIMIC


In [ ]:
import sys
import subprocess
from pathlib import Path

print("Python executable:", sys.executable)

try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd

# Si tu exécutes les cellules dans le désordre, cette cellule doit toujours passer en premier.
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "notebooks").exists() else cwd.parent
raw_csv = PROJECT_ROOT / "data" / "raw" / "patient_vitals_mimic_demo_raw.csv"
demo_root = PROJECT_ROOT / "data" / "raw" / "mimic_iv_demo"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("raw_csv exists:", raw_csv.exists())
print("demo_root exists:", demo_root.exists())

## 1) Vérifier les chemins

In [ ]:
# Sécurité: si tu n’as pas exécuté la cellule d’initialisation, on la refait.
try:
    PROJECT_ROOT
except NameError:
    from pathlib import Path
    cwd = Path.cwd()
    PROJECT_ROOT = cwd if (cwd / "notebooks").exists() else cwd.parent
    raw_csv = PROJECT_ROOT / "data" / "raw" / "patient_vitals_mimic_demo_raw.csv"
    demo_root = PROJECT_ROOT / "data" / "raw" / "mimic_iv_demo"

raw_csv.exists(), demo_root.exists()

## 2) Explorer rapidement la structure du dossier MIMIC (tables disponibles)

In [ ]:
table_files = sorted(demo_root.rglob("*.csv.gz"))
len(table_files), table_files[:10]

## 3) Charger le raw “projet” et regarder les colonnes

In [ ]:
df_raw = pd.read_csv(raw_csv)
df_raw.shape, df_raw.columns.tolist()

In [ ]:
df_raw.head(10)

## 4) Qualité raw (missing, doublons, stats rapides)

In [ ]:
missing_rate = df_raw.isna().mean().sort_values(ascending=False)
missing_rate

In [ ]:
dup_count = int(df_raw.duplicated(subset=["patient_id", "recorded_at"]).sum())
dup_count

In [ ]:
num_cols = [
    "age",
    "temperature_c",
    "systolic_bp",
    "diastolic_bp",
    "spo2",
    "heart_rate",
    "glucose_mg_dl",
    "pain_score",
]
df_raw[num_cols].describe().T

## 5) Quelques vérifications “métier”

- patients avec SpO₂ basse
- patients avec hypotension (systolic < 90)


In [ ]:
df_raw.loc[df_raw["spo2"] < 90].head(10)

In [ ]:
df_raw.loc[df_raw["systolic_bp"] < 90].head(10)